In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   KAGGLE MARCH MANIA 2026 — WOMEN'S CHAMPIONSHIP MODEL v2                  ║
║   Optimised for Apple M1 MacBook Air 8GB                                   ║
║                                                                              ║
║   NEW vs previous model:                                                    ║
║   + Bradley-Terry pairwise strength model                                   ║
║   + Probability calibration (isotonic regression)                           ║
║   + Seed override logic                                                     ║
║   + Upset indicator features (3P dependency, FT rate, tempo conflict)       ║
║   + Optimised ensemble weights via scipy                                    ║
║   + All M1 cores engaged                                                    ║
║                                                                              ║
║   NOTE: Women's model intentionally simpler than men's —                   ║
║   research shows women's bracket has far fewer upsets,                     ║
║   so simpler features work better and avoid overfitting                    ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os
os.environ["OMP_NUM_THREADS"]      = str(os.cpu_count())
os.environ["MKL_NUM_THREADS"]      = str(os.cpu_count())
os.environ["OPENBLAS_NUM_THREADS"] = str(os.cpu_count())
os.environ["VECLIB_MAXIMUM_THREADS"]= str(os.cpu_count())
os.environ["NUMEXPR_NUM_THREADS"]  = str(os.cpu_count())

import warnings, re, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
from pathlib  import Path
from scipy.optimize import minimize

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import log_loss, accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration     import CalibratedClassifierCV
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier

NCORES = os.cpu_count() or 8
t0     = time.time()

DATA_DIR   = Path("/Users/shaurya/Downloads")
OUTPUT_DIR = Path(".")
SEED       = 42
N_TRIALS   = 100
CV_FOLDS   = 5
DECAY_WEEKS= 8
ELO_K      = 20
ELO_BASE   = 1500

SEED_OVERRIDES = {
    (1,16):0.980,(1,15):0.960,(1,14):0.940,(1,13):0.920,
    (2,15):0.950,(2,16):0.970,(2,14):0.910,(2,13):0.880,
    (3,14):0.870,(3,16):0.950,(4,13):0.840,(5,12):0.700,
    (6,11):0.650,(7,10):0.620,(8, 9):0.530,
}

print("=" * 65)
print(f"  MARCH MANIA 2026 — WOMEN'S CHAMPIONSHIP MODEL v2")
print(f"  Running on {NCORES} CPU cores  |  M1 optimised")
print("=" * 65)

def load(f): return pd.read_csv(DATA_DIR / f)
def elapsed(): return f"{(time.time()-t0)/60:.1f}m"
def _ds(df):   return df.drop(columns=["Season"], errors="ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[1/8] Loading data …  {elapsed()}")
w_reg_det  = load("WRegularSeasonDetailedResults.csv")
w_reg_cmp  = load("WRegularSeasonCompactResults.csv")
w_trn_cmp  = load("WNCAATourneyCompactResults.csv")
w_seeds    = load("WNCAATourneySeeds.csv")
w_conf     = load("WTeamConferences.csv")
w_conf_trn = load("WConferenceTourneyGames.csv")
sub_s2     = load("SampleSubmissionStage2.csv")
print(f"   reg_det={len(w_reg_det):,}  trn_cmp={len(w_trn_cmp):,}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. ELO
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[2/8] ELO …  {elapsed()}")

def compute_elo(cmp):
    elo,rows={},[]
    for _,r in cmp.sort_values(["Season","DayNum"]).iterrows():
        w,l=int(r.WTeamID),int(r.LTeamID)
        elo.setdefault(w,ELO_BASE); elo.setdefault(l,ELO_BASE)
        ew   =1.0/(1.0+10**((elo[l]-elo[w])/400.0))
        mult =np.log1p(abs(r.WScore-r.LScore))/np.log1p(10)
        delta=ELO_K*mult*(1-ew)
        rows+=[{"Season":r.Season,"TeamID":w,"ELO":elo[w]},
               {"Season":r.Season,"TeamID":l,"ELO":elo[l]}]
        elo[w]+=delta; elo[l]-=delta
    tmp=pd.DataFrame(rows)
    return (tmp.groupby(["Season","TeamID"])["ELO"]
               .last().reset_index()
               .rename(columns={"ELO":"ELO_rating"}))

elo_df=compute_elo(pd.concat([w_reg_cmp,w_trn_cmp],ignore_index=True))

# ─────────────────────────────────────────────────────────────────────────────
# 3. ADVANCED STATS + UPSET INDICATORS
# ─────────────────────────────────────────────────────────────────────────────
print(f"[3/8] Advanced stats …  {elapsed()}")

def make_stats(det):
    def poss(fga,fta,orb,to): return fga-orb+to+0.475*fta
    rows=[]
    for _,r in det.iterrows():
        wp=poss(r.WFGA,r.WFTA,r.WOR,r.WTO)
        lp=poss(r.LFGA,r.LFTA,r.LOR,r.LTO)
        p=(wp+lp)/2+1e-6
        for px,ox,wn in [("W","L",1),("L","W",0)]:
            sf=r[f"{px}Score"]; sa=r[f"{ox}Score"]
            rows.append(dict(
                Season=r.Season, TeamID=r[f"{px}TeamID"], Won=wn,
                Diff=sf-sa,
                FGPct =r[f"{px}FGM"]/max(r[f"{px}FGA"],1),
                FG3Pct=r[f"{px}FGM3"]/max(r[f"{px}FGA3"],1),
                FTPct =r[f"{px}FTM"]/max(r[f"{px}FTA"],1),
                AstTOR=r[f"{px}Ast"]/max(r[f"{px}TO"],1),
                RebPct=(r[f"{px}OR"]+r[f"{px}DR"])/
                        max(r[f"{px}OR"]+r[f"{px}DR"]+r[f"{ox}OR"]+r[f"{ox}DR"],1),
                OffRtg=sf/p*100, DefRtg=sa/p*100,
                NetRtg=(sf-sa)/p*100, Pace=p,
                AvgStl=r[f"{px}Stl"], AvgBlk=r[f"{px}Blk"], AvgTO=r[f"{px}TO"],
                ThreePtDep=r[f"{px}FGM3"]*3/max(sf,1),
                FTRate=r[f"{px}FTA"]/max(r[f"{px}FGA"],1),
                TempoPace=poss(r[f"{px}FGA"],r[f"{px}FTA"],r[f"{px}OR"],r[f"{px}TO"]),
            ))
    df=pd.DataFrame(rows)
    return df.groupby(["Season","TeamID"]).agg(
        WinRate=("Won","mean"), AvgDiff=("Diff","mean"),
        FGPct=("FGPct","mean"), FG3Pct=("FG3Pct","mean"),
        FTPct=("FTPct","mean"), AstTOR=("AstTOR","mean"),
        RebPct=("RebPct","mean"), OffRtg=("OffRtg","mean"),
        DefRtg=("DefRtg","mean"), NetRtg=("NetRtg","mean"),
        Pace=("Pace","mean"), AvgStl=("AvgStl","mean"),
        AvgBlk=("AvgBlk","mean"), AvgTO=("AvgTO","mean"),
        GP=("Won","count"),
        ThreePtDep=("ThreePtDep","mean"),
        FTRate=("FTRate","mean"),
        TempoPace=("TempoPace","mean"),
    ).reset_index()

stats_df=make_stats(w_reg_det)

# ─────────────────────────────────────────────────────────────────────────────
# 4. MOMENTUM + DECAY
# ─────────────────────────────────────────────────────────────────────────────
print(f"[4/8] Momentum …  {elapsed()}")

def make_momentum(cmp,last_n=10):
    rw=cmp[["Season","DayNum","WTeamID","WScore","LScore"]].copy()
    rw.columns=["Season","DayNum","TeamID","For","Against"]; rw["Won"]=1
    rl=cmp[["Season","DayNum","LTeamID","LScore","WScore"]].copy()
    rl.columns=["Season","DayNum","TeamID","For","Against"]; rl["Won"]=0
    lng=pd.concat([rw,rl]).sort_values(["Season","TeamID","DayNum"])
    lng["Diff"]=lng["For"]-lng["Against"]
    reg=lng[lng["DayNum"]<=132]
    out=[]
    for (ssn,tid),g in reg.groupby(["Season","TeamID"]):
        g=g.sort_values("DayNum"); last=g.tail(last_n)
        if len(last)==0: continue
        mx=last["DayNum"].max()
        da=(mx-last["DayNum"]).values.astype(float)
        wd=np.exp(-np.log(2)/(DECAY_WEEKS*7)*da)
        tw=wd.sum()+1e-9
        mwr=(last["Won"].values*wd).sum()/tw
        mdf=(last["Diff"].values*wd).sum()/tw
        wl=g["Won"].tolist(); st=0
        for v in reversed(wl):
            if st==0: st=1 if v else -1
            elif st>0 and v: st+=1
            elif st<0 and not v: st-=1
            else: break
        l5=g.tail(5)
        hot=float(l5["Won"].mean()-g["Won"].mean()) if len(l5) else 0.
        out.append({"Season":ssn,"TeamID":tid,
                    "MomWR":mwr,"MomDiff":mdf,"Streak":st,"HotCold":hot})
    return pd.DataFrame(out)

mom_df=make_momentum(w_reg_cmp)

# ─────────────────────────────────────────────────────────────────────────────
# 5. BRADLEY-TERRY
# ─────────────────────────────────────────────────────────────────────────────
print(f"[5/8] Bradley-Terry …  {elapsed()}")

def compute_bt(cmp_df,seasons):
    bt_results=[]
    for ssn in seasons:
        games=cmp_df[cmp_df["Season"]==ssn]
        if len(games)==0: continue
        teams=sorted(set(games["WTeamID"].tolist()+games["LTeamID"].tolist()))
        n=len(teams)
        if n<3: continue
        idx={t:i for i,t in enumerate(teams)}
        wins=np.zeros(n); loss_arr=np.zeros((n,n))
        for _,r in games.iterrows():
            wi=idx[int(r.WTeamID)]; li=idx[int(r.LTeamID)]
            wins[wi]+=1; loss_arr[wi,li]+=1
        strength=np.ones(n)
        for _ in range(50):
            new_s=np.zeros(n)
            for i in range(n):
                denom=sum((loss_arr[i,j]+loss_arr[j,i])/(strength[i]+strength[j])
                          for j in range(n) if i!=j and (loss_arr[i,j]+loss_arr[j,i])>0)
                if denom>0: new_s[i]=wins[i]/denom
            new_s=new_s/(new_s.mean()+1e-9)
            if np.max(np.abs(new_s-strength))<1e-6: break
            strength=new_s
        for i,t in enumerate(teams):
            bt_results.append({"Season":ssn,"TeamID":t,"BTStrength":strength[i]})
    return pd.DataFrame(bt_results)

bt_df=compute_bt(pd.concat([w_reg_cmp,w_trn_cmp],ignore_index=True),
                 list(range(2013,2027)))

# ─────────────────────────────────────────────────────────────────────────────
# 6. AUXILIARY
# ─────────────────────────────────────────────────────────────────────────────
def parse_seed(s):
    m=re.search(r'\d+',str(s)); return int(m.group()) if m else 16

w_seeds["SeedNum"]=w_seeds["Seed"].apply(parse_seed)
seed_df  =w_seeds[["Season","TeamID","SeedNum"]].copy()
conf_mem =w_conf[["Season","TeamID","ConfAbbrev"]].copy()
conf_nr  =(stats_df[["Season","TeamID","NetRtg"]]
           .merge(conf_mem,on=["Season","TeamID"],how="left")
           .groupby(["Season","ConfAbbrev"])["NetRtg"]
           .mean().reset_index()
           .rename(columns={"NetRtg":"ConfNetRtg"}))
ctw_df   =(w_conf_trn.groupby(["Season","WTeamID"]).size()
                     .reset_index()
                     .rename(columns={"WTeamID":"TeamID",0:"CTW"}))
POWER_CONF={"acc","big_east","big_ten","big_twelve","sec","pac_twelve","pac_ten"}
conf_mem["PowerConf"]=conf_mem["ConfAbbrev"].isin(POWER_CONF).astype(int)
power_df=conf_mem[["Season","TeamID","PowerConf"]].copy()

# ─────────────────────────────────────────────────────────────────────────────
# 7. PROFILE + MATRIX
# ─────────────────────────────────────────────────────────────────────────────
def profile_for_season(ssn):
    def filt(df): return _ds(df[df["Season"]==ssn].copy())
    base=filt(conf_mem)[["TeamID"]].drop_duplicates()
    st=filt(stats_df); el=filt(elo_df); mo=filt(mom_df)
    se=filt(seed_df);  bt=filt(bt_df)
    cn=filt(conf_mem)[["TeamID","ConfAbbrev"]]
    nr=filt(conf_nr)[["ConfAbbrev","ConfNetRtg"]]
    ct=filt(ctw_df); pw=filt(power_df)
    cn2=cn.merge(nr,on="ConfAbbrev",how="left")[["TeamID","ConfNetRtg"]]
    p=(base
       .merge(st, on="TeamID",how="left")
       .merge(el, on="TeamID",how="left")
       .merge(mo, on="TeamID",how="left")
       .merge(se, on="TeamID",how="left")
       .merge(bt, on="TeamID",how="left")
       .merge(cn2,on="TeamID",how="left")
       .merge(ct, on="TeamID",how="left")
       .merge(pw, on="TeamID",how="left"))
    p["CTW"]       =p["CTW"].fillna(0)
    p["SeedNum"]   =p["SeedNum"].fillna(16)
    p["Streak"]    =p["Streak"].fillna(0)
    p["BTStrength"]=p["BTStrength"].fillna(1.0)
    p["PowerConf"] =p["PowerConf"].fillna(0)
    return p.fillna(0)

_sample  =profile_for_season(2023)
SKIP     ={"TeamID","ConfAbbrev"}
FEAT_COLS=[c for c in _sample.columns if c not in SKIP]

def matchup_season(ssn,pairs,labels=None):
    prof=profile_for_season(ssn).set_index("TeamID")
    fc=[c for c in FEAT_COLS if c in prof.columns]
    nf=len(fc); zero=np.zeros(nf)
    X,y=[],[]
    for i,(t1,t2) in enumerate(pairs):
        p1=np.nan_to_num(prof.loc[t1,fc].values.astype(float) if t1 in prof.index else zero.copy())
        p2=np.nan_to_num(prof.loc[t2,fc].values.astype(float) if t2 in prof.index else zero.copy())
        tc=abs(p1[fc.index("TempoPace")]-p2[fc.index("TempoPace")]) if "TempoPace" in fc else 0
        tg=abs(p1[fc.index("ThreePtDep")]-p2[fc.index("ThreePtDep")]) if "ThreePtDep" in fc else 0
        X.append(np.concatenate([p1-p2,p1,p2,[tc,tg]]))
        if labels is not None: y.append(labels[i])
    return np.array(X),(np.array(y) if labels is not None else None)

def build_matrix(mu_df):
    has_label="Label" in mu_df.columns
    Xp,yp=[],[]
    for ssn,grp in mu_df.groupby("Season"):
        pairs=list(zip(grp["Team1"].astype(int),grp["Team2"].astype(int)))
        labels=grp["Label"].tolist() if has_label else None
        Xs,ys=matchup_season(ssn,pairs,labels)
        Xp.append(Xs)
        if has_label: yp.append(ys)
    X=np.vstack(Xp)
    y=np.concatenate(yp) if has_label else None
    return X,y

def tourney_mu(df):
    rows=[]
    for _,r in df.iterrows():
        w,l=int(r.WTeamID),int(r.LTeamID)
        t1,t2=(w,l) if w<l else (l,w)
        rows.append({"Season":r.Season,"Team1":t1,"Team2":t2,
                     "Label":1 if t1==w else 0})
    return pd.DataFrame(rows)

def reg_mu(df,n=600):
    rows=[]
    for _,r in df.iterrows():
        w,l=int(r.WTeamID),int(r.LTeamID)
        t1,t2=(w,l) if w<l else (l,w)
        rows.append({"Season":r.Season,"Team1":t1,"Team2":t2,
                     "Label":1 if t1==w else 0})
    d=pd.DataFrame(rows)
    return (d.groupby("Season",group_keys=False)
             .apply(lambda x:x.sample(min(n,len(x)),random_state=SEED))
             .reset_index(drop=True))

print(f"\n[6/8] Building feature matrices …  {elapsed()}")
TRAIN_REG=list(range(2013,2025)); TRAIN_TRN=list(range(2013,2024))
VALID_TRN=[2024]; VALID_REG=[2025]; TEST_TRN=[2025]

train_mu=pd.concat([
    reg_mu(w_reg_cmp[w_reg_cmp["Season"].isin(TRAIN_REG)]),
    tourney_mu(w_trn_cmp[w_trn_cmp["Season"].isin(TRAIN_TRN)])
],ignore_index=True)
valid_mu=pd.concat([
    tourney_mu(w_trn_cmp[w_trn_cmp["Season"].isin(VALID_TRN)]),
    reg_mu(w_reg_cmp[w_reg_cmp["Season"].isin(VALID_REG)])
],ignore_index=True)
test_mu=tourney_mu(w_trn_cmp[w_trn_cmp["Season"].isin(TEST_TRN)])

print(f"   Train={len(train_mu):,}  Valid={len(valid_mu):,}  Test={len(test_mu):,}")
X_tr,y_tr=build_matrix(train_mu)
X_va,y_va=build_matrix(valid_mu)
X_te,y_te=build_matrix(test_mu)
print(f"   Feature dims: {X_tr.shape[1]}")

scaler=StandardScaler()
Xs_tr=scaler.fit_transform(X_tr)
Xs_va=scaler.transform(X_va)
Xs_te=scaler.transform(X_te)

# ─────────────────────────────────────────────────────────────────────────────
# 8. OPTUNA + CALIBRATION + ENSEMBLE
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[7/8] Optuna tuning …  {elapsed()}")
cv=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=SEED)

print(f"  XGBoost ({N_TRIALS} trials) …")
def obj_xgb(trial):
    p=dict(n_estimators=trial.suggest_int("n_estimators",100,800),
           max_depth=trial.suggest_int("max_depth",3,10),
           learning_rate=trial.suggest_float("learning_rate",0.005,0.3,log=True),
           subsample=trial.suggest_float("subsample",0.5,1.0),
           colsample_bytree=trial.suggest_float("colsample_bytree",0.4,1.0),
           min_child_weight=trial.suggest_int("min_child_weight",1,20),
           reg_alpha=trial.suggest_float("reg_alpha",1e-4,10.0,log=True),
           reg_lambda=trial.suggest_float("reg_lambda",1e-4,10.0,log=True),
           gamma=trial.suggest_float("gamma",0.0,5.0))
    m=XGBClassifier(**p,objective="binary:logistic",eval_metric="logloss",
                    use_label_encoder=False,random_state=SEED,n_jobs=NCORES,verbosity=0)
    return -cross_val_score(m,X_tr,y_tr,cv=cv,scoring="neg_log_loss",n_jobs=NCORES).mean()

xgb_study=optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(obj_xgb,n_trials=N_TRIALS,show_progress_bar=True)

print(f"  LightGBM ({N_TRIALS} trials) …")
def obj_lgb(trial):
    p=dict(n_estimators=trial.suggest_int("n_estimators",100,800),
           max_depth=trial.suggest_int("max_depth",3,10),
           learning_rate=trial.suggest_float("learning_rate",0.005,0.3,log=True),
           num_leaves=trial.suggest_int("num_leaves",15,127),
           subsample=trial.suggest_float("subsample",0.5,1.0),
           colsample_bytree=trial.suggest_float("colsample_bytree",0.4,1.0),
           min_child_samples=trial.suggest_int("min_child_samples",5,50),
           reg_alpha=trial.suggest_float("reg_alpha",1e-4,10.0,log=True),
           reg_lambda=trial.suggest_float("reg_lambda",1e-4,10.0,log=True))
    m=LGBMClassifier(**p,objective="binary",metric="binary_logloss",
                     random_state=SEED,n_jobs=NCORES,verbose=-1)
    return -cross_val_score(m,X_tr,y_tr,cv=cv,scoring="neg_log_loss",n_jobs=NCORES).mean()

lgb_study=optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(obj_lgb,n_trials=N_TRIALS,show_progress_bar=True)

print(f"  LogReg ({N_TRIALS} trials) …")
def obj_lr(trial):
    C=trial.suggest_float("C",1e-4,100.0,log=True)
    pen=trial.suggest_categorical("penalty",["l1","l2"])
    mit=trial.suggest_int("max_iter",200,2000)
    m=LogisticRegression(C=C,penalty=pen,solver="liblinear",max_iter=mit,random_state=SEED)
    return -cross_val_score(m,Xs_tr,y_tr,cv=cv,scoring="neg_log_loss",n_jobs=NCORES).mean()

lr_study=optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=SEED))
lr_study.optimize(obj_lr,n_trials=N_TRIALS,show_progress_bar=True)

print("  Fitting & calibrating …")
xgb_base=XGBClassifier(**xgb_study.best_params,objective="binary:logistic",
                        eval_metric="logloss",use_label_encoder=False,
                        random_state=SEED,n_jobs=NCORES,verbosity=0)
lgb_base=LGBMClassifier(**lgb_study.best_params,objective="binary",
                         metric="binary_logloss",random_state=SEED,n_jobs=NCORES,verbose=-1)
bp=lr_study.best_params
lr_base=LogisticRegression(C=bp["C"],penalty=bp["penalty"],solver="liblinear",
                            max_iter=bp["max_iter"],random_state=SEED)
xgb_base.fit(X_tr,y_tr); lgb_base.fit(X_tr,y_tr); lr_base.fit(Xs_tr,y_tr)

xgb_cal=CalibratedClassifierCV(xgb_base,method="isotonic",cv=3)
lgb_cal=CalibratedClassifierCV(lgb_base,method="isotonic",cv=3)
lr_cal =CalibratedClassifierCV(lr_base, method="isotonic",cv=3)
xgb_cal.fit(X_tr,y_tr); lgb_cal.fit(X_tr,y_tr); lr_cal.fit(Xs_tr,y_tr)

p_xgb_va=xgb_cal.predict_proba(X_va)[:,1]
p_lgb_va=lgb_cal.predict_proba(X_va)[:,1]
p_lr_va =lr_cal.predict_proba(Xs_va)[:,1]

def neg_ll(w):
    w=np.abs(w)/np.abs(w).sum()
    return log_loss(y_va,w[0]*p_xgb_va+w[1]*p_lgb_va+w[2]*p_lr_va)

res=minimize(neg_ll,[1/3,1/3,1/3],method="Nelder-Mead")
raw=np.abs(res.x); wts=raw/raw.sum()
w1,w2,w3=wts[0],wts[1],wts[2]
print(f"   Weights  XGB={w1:.3f}  LGB={w2:.3f}  LR={w3:.3f}")

def ens(X,Xs):
    return (w1*xgb_cal.predict_proba(X)[:,1]+
            w2*lgb_cal.predict_proba(X)[:,1]+
            w3*lr_cal.predict_proba(Xs)[:,1])

# ─────────────────────────────────────────────────────────────────────────────
# SEED OVERRIDE
# ─────────────────────────────────────────────────────────────────────────────
def apply_seed_overrides(pred,team1,team2,season):
    seed_map={}
    ssn_seeds=w_seeds[w_seeds["Season"]==season]
    for _,r in ssn_seeds.iterrows():
        seed_map[int(r.TeamID)]=int(r.SeedNum)
    adjusted=pred.copy()
    for i,(t1,t2) in enumerate(zip(team1,team2)):
        s1=seed_map.get(int(t1),8); s2=seed_map.get(int(t2),8)
        key=(min(s1,s2),max(s1,s2))
        if key in SEED_OVERRIDES:
            override=SEED_OVERRIDES[key]
            if s1<s2: adjusted[i]=max(adjusted[i],override)
            else:     adjusted[i]=min(adjusted[i],1-override)
    return adjusted

vl_ens=log_loss(y_va,ens(X_va,Xs_va))
tl_ens=log_loss(y_te,ens(X_te,Xs_te))
acc_te=accuracy_score(y_te,(ens(X_te,Xs_te)>=0.5).astype(int))
auc_te=roc_auc_score(y_te,ens(X_te,Xs_te))

print(f"\n   ENS  Val={vl_ens:.5f}  Test={tl_ens:.5f}")
print(f"   ENS  Accuracy={acc_te*100:.2f}%  AUC={auc_te:.5f}")

# ─────────────────────────────────────────────────────────────────────────────
# PREDICT 2026
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[8/8] Predicting 2026 …  {elapsed()}")
sub=sub_s2.copy()
parts=sub["ID"].str.split("_",expand=True).astype(int)
sub["Season"]=parts[0]; sub["Team1"]=parts[1]; sub["Team2"]=parts[2]
sub_w=sub[(sub["Team1"]>=3000)&(sub["Team2"]>=3000)].copy()
print(f"   Women's rows: {len(sub_w):,}")

X_pred,_=build_matrix(sub_w[["Season","Team1","Team2"]])
Xs_pred =scaler.transform(X_pred)
raw_pred=ens(X_pred,Xs_pred)
raw_pred=apply_seed_overrides(raw_pred,sub_w["Team1"].values,sub_w["Team2"].values,2026)
sub_w=sub_w.copy()
sub_w["Pred"]=raw_pred.clip(0.025,0.975)

out=OUTPUT_DIR/"submission_womens_2026_v2.csv"
sub_w[["ID","Pred"]].to_csv(out,index=False)

print("\n"+"="*65)
print("  FINAL SUMMARY — WOMEN'S CHAMPIONSHIP MODEL v2")
print("="*65)
print(f"\n  ── Hyperparameters ──────────────────────────────────")
print(f"  XGBoost:"); [print(f"    {k:<25} → {v}") for k,v in sorted(xgb_study.best_params.items())]
print(f"\n  LightGBM:"); [print(f"    {k:<25} → {v}") for k,v in sorted(lgb_study.best_params.items())]
print(f"\n  LogReg:"); [print(f"    {k:<25} → {v}") for k,v in sorted(lr_study.best_params.items())]
print(f"\n  ── Performance ──────────────────────────────────────")
print(f"  {'Model':<12} {'Val LL':>9} {'Test LL':>9} {'Accuracy':>10} {'AUC':>9}")
print("  "+"-"*52)
for name,pv,pt in [
    ("XGBoost", p_xgb_va, xgb_cal.predict_proba(X_te)[:,1]),
    ("LightGBM",p_lgb_va, lgb_cal.predict_proba(X_te)[:,1]),
    ("LogReg",  p_lr_va,  lr_cal.predict_proba(Xs_te)[:,1]),
    ("Ensemble",ens(X_va,Xs_va),ens(X_te,Xs_te))]:
    acc=accuracy_score(y_te,(pt>=0.5).astype(int))*100
    auc=roc_auc_score(y_te,pt)
    print(f"  {name:<12} {log_loss(y_va,pv):>9.5f} {log_loss(y_te,pt):>9.5f} "
          f"{acc:>9.2f}% {auc:>9.5f}")
print(f"\n  Weights: XGB={w1:.3f} LGB={w2:.3f} LR={w3:.3f}")
print(f"  Runtime: {elapsed()}")
print(f"  Rows: {len(sub_w):,}   ✅ Saved → {out}")
print("="*65)
print(sub_w[["ID","Pred"]].head(10).to_string(index=False))

In [2]:
"""
MARCH MANIA 2026 — WOMEN'S  |  EXACT REPLICA OF 89% MODEL ARCHITECTURE
========================================================================
Same architecture as the men's 89% model.
Women use WNCAATourneyDetailedResults.csv instead.
pip install lightgbm scikit-learn pandas numpy
"""

import os
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import warnings, re, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score, brier_score_loss
from lightgbm import LGBMClassifier

t0 = time.time()

DATA_DIR = Path("/Users/shaurya/Downloads")
SEED     = 42
np.random.seed(SEED)

BEST_LGBM = {
    "num_leaves"        : 170,
    "max_depth"         : 12,
    "learning_rate"     : 0.005,
    "n_estimators"      : 1308 + 200,
    "min_child_samples" : 5,
    "subsample"         : 0.5,
    "colsample_bytree"  : 0.5,
    "reg_alpha"         : 0.0,
    "reg_lambda"        : 0.0,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : 1,
    "verbose"           : -1,
}

TRAIN_SEASONS     = list(range(2010, 2025))
VAL_SEASONS_TOURN = [2024]
VAL_SEASONS_REG   = [2025]
TEST_SEASONS      = [2025]
ALL_SEASONS       = TRAIN_SEASONS + [2024, 2025]

ELO_K        = 20
ELO_BASE     = 1500
ELO_HOME_ADV = 100

# Women: stronger seed overrides
SEED_OVERRIDES = {
    (1,16):0.985,(1,15):0.965,(1,14):0.945,(1,13):0.925,
    (2,15):0.955,(2,16):0.975,(3,14):0.875,(4,13):0.850,
    (5,12):0.720,(6,11):0.680,(7,10):0.650,(8, 9):0.540,
}

def mins(): return f"{(time.time()-t0)/60:.1f}m"
def load(f): return pd.read_csv(DATA_DIR / f)

print("="*60)
print("  MARCH MANIA 2026 — WOMEN'S  (89% EXACT REPLICA)")
print("="*60)

# ── LOAD ──────────────────────────────────────────────────────────────────────
print(f"\n[1/6] Loading …  {mins()}")
reg_df     = load("WRegularSeasonDetailedResults.csv")
tourney_df = load("WNCAATourneyDetailedResults.csv")
seeds_df   = load("WNCAATourneySeeds.csv")
sub_s2     = load("SampleSubmissionStage2.csv")

reg_df     = reg_df[reg_df["Season"].isin(ALL_SEASONS)].copy()
tourney_df = tourney_df[tourney_df["Season"].isin(ALL_SEASONS)].copy()
print(f"   reg={len(reg_df):,}  tourney={len(tourney_df):,}")

# ── ELO ───────────────────────────────────────────────────────────────────────
print(f"[2/6] ELO …  {mins()}")

def compute_elo(reg_df, tourney_df):
    all_games = pd.concat([
        reg_df[["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]],
        tourney_df[["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]]
    ]).sort_values(["Season","DayNum"]).reset_index(drop=True)

    elo = {}
    season_end = {}

    for season in sorted(all_games["Season"].unique()):
        for tid in list(elo):
            elo[tid] = ELO_BASE * 0.25 + elo[tid] * 0.75

        for _, r in all_games[all_games["Season"]==season].iterrows():
            w, l = int(r["WTeamID"]), int(r["LTeamID"])
            rw = elo.get(w, ELO_BASE); rl = elo.get(l, ELO_BASE)
            loc = r["WLoc"]
            rw_adj = rw+ELO_HOME_ADV if loc=="H" else (rw-ELO_HOME_ADV if loc=="A" else rw)
            mov    = min(abs(int(r["WScore"])-int(r["LScore"])), 20)
            mov_mult = np.log1p(mov)*(2.2/((rw_adj-rl)*0.001+2.2))
            exp_w  = 1/(1+10**((rl-rw_adj)/400))
            delta  = ELO_K*mov_mult*(1-exp_w)
            elo[w] = rw+delta; elo[l] = rl-delta

        for tid,rat in elo.items():
            season_end[(season,tid)] = rat

    return season_end

elo_dict = compute_elo(reg_df, tourney_df)

# ── TEAM FEATURES ─────────────────────────────────────────────────────────────
print(f"[3/6] Team features …  {mins()}")

STAT_COLS = ["FGM","FGA","FGM3","FGA3","FTM","FTA","OR","DR","Ast","TO","Stl","Blk","PF"]

def explode_games(games):
    rw = {"WTeamID":"TeamID","LTeamID":"OppID","WScore":"PtsFor","LScore":"PtsAgainst",
          **{f"W{s}":s for s in STAT_COLS}, **{f"L{s}":f"Opp{s}" for s in STAT_COLS}}
    rl = {"LTeamID":"TeamID","WTeamID":"OppID","LScore":"PtsFor","WScore":"PtsAgainst",
          **{f"L{s}":s for s in STAT_COLS}, **{f"W{s}":f"Opp{s}" for s in STAT_COLS}}
    keep = ["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst"]+\
           STAT_COLS+[f"Opp{s}" for s in STAT_COLS]
    w=games.rename(columns=rw); w["Win"]=1
    l=games.rename(columns=rl); l["Win"]=0
    return pd.concat([w[keep+["Win"]],l[keep+["Win"]]],ignore_index=True)

def team_season_features(reg_df, tourney_df, seasons):
    both=pd.concat([reg_df,tourney_df])[lambda d: d["Season"].isin(seasons)]
    tg=explode_games(both); eps=1e-6
    agg=tg.groupby(["Season","TeamID"]).agg(
        GP=("Win","count"),Wins=("Win","sum"),
        PtsFor=("PtsFor","mean"),PtsAgainst=("PtsAgainst","mean"),
        FGM=("FGM","mean"),FGA=("FGA","mean"),
        FGM3=("FGM3","mean"),FGA3=("FGA3","mean"),
        FTM=("FTM","mean"),FTA=("FTA","mean"),
        OR=("OR","mean"),DR=("DR","mean"),
        Ast=("Ast","mean"),TO=("TO","mean"),
        Stl=("Stl","mean"),Blk=("Blk","mean"),PF=("PF","mean"),
        OppFGM=("OppFGM","mean"),OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"),OppFGA3=("OppFGA3","mean"),
        OppOR=("OppOR","mean"),OppDR=("OppDR","mean"),
    ).reset_index()
    agg["WinPct"]   =agg["Wins"]/(agg["GP"]+eps)
    agg["FGPct"]    =agg["FGM"]/(agg["FGA"]+eps)
    agg["FG3Pct"]   =agg["FGM3"]/(agg["FGA3"]+eps)
    agg["FTPct"]    =agg["FTM"]/(agg["FTA"]+eps)
    agg["OppFGPct"] =agg["OppFGM"]/(agg["OppFGA"]+eps)
    agg["eFGPct"]   =(agg["FGM"]+0.5*agg["FGM3"])/(agg["FGA"]+eps)
    agg["OppEFGPct"]=(agg["OppFGM"]+0.5*agg["OppFGM3"])/(agg["OppFGA"]+eps)
    poss            =agg["FGA"]-agg["OR"]+agg["TO"]+0.44*agg["FTA"]
    agg["TOVRate"]  =agg["TO"]/(poss+eps)
    agg["ORBRate"]  =agg["OR"]/(agg["OR"]+agg["OppDR"]+eps)
    agg["FTRate"]   =agg["FTM"]/(agg["FGA"]+eps)
    agg["DRBRate"]  =agg["DR"]/(agg["DR"]+agg["OppOR"]+eps)
    agg["Pace"]     =poss/(agg["GP"]+eps)
    agg["NetRtg"]   =agg["PtsFor"]-agg["PtsAgainst"]
    agg["OffRtg"]   =agg["PtsFor"]/(poss+eps)*100
    agg["DefRtg"]   =agg["PtsAgainst"]/(poss+eps)*100
    agg["NetRtg2"]  =agg["OffRtg"]-agg["DefRtg"]
    agg["AstTO"]    =agg["Ast"]/(agg["TO"]+eps)
    agg["StlBlk"]   =(agg["Stl"]+agg["Blk"])/(agg["OppFGA"]+eps)
    return agg

def recent_form(reg_df,seasons,n=10):
    tg=explode_games(reg_df[reg_df["Season"].isin(seasons)])
    tg=tg.sort_values(["Season","TeamID","DayNum"])
    rec=tg.groupby(["Season","TeamID"]).tail(n)
    rf=rec.groupby(["Season","TeamID"]).agg(
        RecentWinPct=("Win","mean"),
        RecentPtsFor=("PtsFor","mean"),
        RecentPtsAgainst=("PtsAgainst","mean"),
    ).reset_index()
    rf["RecentNetPts"]=rf["RecentPtsFor"]-rf["RecentPtsAgainst"]
    return rf

team_stats = team_season_features(reg_df, tourney_df, ALL_SEASONS)
recent     = recent_form(reg_df, ALL_SEASONS)

# ── MATCHUPS ──────────────────────────────────────────────────────────────────
print(f"[4/6] Matchups …  {mins()}")

def parse_seed(s):
    if pd.isna(s): return 16
    m=re.search(r"\d+",str(s)); return int(m.group()) if m else 16

seed_feat=seeds_df.copy()
seed_feat["SeedNum"]=seed_feat["Seed"].apply(parse_seed)
seed_feat=seed_feat[["Season","TeamID","SeedNum"]]

def build_matchup(games,team_stats,recent,elo_dict,seed_feat):
    df=games.copy()
    df["TeamA"]=df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"]=df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"]=(df["WTeamID"]==df["TeamA"]).astype(int)

    def mg(df,stats,prefix,side):
        cols=[c for c in stats.columns if c not in ["Season","TeamID"]]
        s=stats.rename(columns={c:f"{prefix}_{c}" for c in cols})
        return df.merge(s,left_on=["Season",side],
                        right_on=["Season","TeamID"],how="left")\
                 .drop(columns=["TeamID"],errors="ignore")

    df=mg(df,team_stats,"A","TeamA")
    df=mg(df,team_stats,"B","TeamB")
    df=mg(df,recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                       "RecentPtsFor","RecentPtsAgainst"]],"A","TeamA")
    df=mg(df,recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                       "RecentPtsFor","RecentPtsAgainst"]],"B","TeamB")
    df=mg(df,seed_feat,"A","TeamA")
    df=mg(df,seed_feat,"B","TeamB")

    df["A_ELO"]=df.apply(lambda r:elo_dict.get((r["Season"],int(r["TeamA"])),ELO_BASE),axis=1)
    df["B_ELO"]=df.apply(lambda r:elo_dict.get((r["Season"],int(r["TeamB"])),ELO_BASE),axis=1)

    feature_cols=[]
    for ca in [c for c in df.columns if c.startswith("A_")]:
        base=ca[2:]; cb=f"B_{base}"
        if cb in df.columns:
            dc=f"diff_{base}"; df[dc]=df[ca]-df[cb]
            feature_cols.extend([dc,ca,cb])

    df["ELO_WinProb"]=1/(1+10**((df["B_ELO"]-df["A_ELO"])/400))
    feature_cols=list(dict.fromkeys(feature_cols+["ELO_WinProb"]))
    return df, feature_cols

reg_df["IsTourney"]=False
tourney_df["IsTourney"]=True
all_games=pd.concat([reg_df,tourney_df],ignore_index=True)

matchup_df, feature_cols = build_matchup(
    all_games, team_stats, recent, elo_dict, seed_feat)
print(f"   Rows={len(matchup_df):,}  Features={len(feature_cols)}")

# ── SPLITS ────────────────────────────────────────────────────────────────────
train_mask=matchup_df["Season"].isin(TRAIN_SEASONS)
val_mask=(
    (matchup_df["Season"].isin(VAL_SEASONS_TOURN)&matchup_df["IsTourney"])|
    (matchup_df["Season"].isin(VAL_SEASONS_REG)  &~matchup_df["IsTourney"])
)
test_mask=matchup_df["Season"].isin(TEST_SEASONS)&matchup_df["IsTourney"]

X_tr=matchup_df.loc[train_mask,feature_cols].fillna(0)
y_tr=matchup_df.loc[train_mask,"Label"]
X_val=matchup_df.loc[val_mask,feature_cols].fillna(0)
y_val=matchup_df.loc[val_mask,"Label"]
X_te=matchup_df.loc[test_mask,feature_cols].fillna(0)
y_te=matchup_df.loc[test_mask,"Label"]
X_full=pd.concat([X_tr,X_val]); y_full=pd.concat([y_tr,y_val])
print(f"   Train={len(X_tr):,}  Val={len(X_val):,}  Test={len(X_te):,}")

# ── TRAIN ─────────────────────────────────────────────────────────────────────
print(f"\n[5/6] Training …  {mins()}")
model=LGBMClassifier(**BEST_LGBM)
model.fit(X_full,y_full)

p_val=model.predict_proba(X_val)[:,1]
p_test=model.predict_proba(X_te)[:,1]
val_ll=log_loss(y_val,p_val)
test_ll=log_loss(y_te,p_test)
test_auc=roc_auc_score(y_te,p_test)
test_bri=brier_score_loss(y_te,p_test)
test_acc=accuracy_score(y_te,(p_test>0.5).astype(int))
n_correct=int(((p_test>0.5)==y_te.values).sum())
print(f"\n   [Validation]    LogLoss : {val_ll:.5f}")
print(f"   [2025 Tourney]  LogLoss : {test_ll:.5f}")
print(f"   [2025 Tourney]  AUC     : {test_auc:.4f}")
print(f"   [2025 Tourney]  Accuracy: {test_acc:.4f}  ({n_correct}/{len(y_te)})")

# ── PREDICT 2026 ──────────────────────────────────────────────────────────────
print(f"\n[6/6] Predicting 2026 …  {mins()}")
sub=sub_s2.copy()
sub[["Season","Team1","Team2"]]=(sub["ID"].str.split("_",expand=True)
                                          .astype(int)
                                          .rename(columns={0:"Season",1:"Team1",2:"Team2"}))
sub_w=sub[(sub["Team1"]>=3000)&(sub["Team2"]>=3000)].copy()
print(f"   Women's rows: {len(sub_w):,}")

pred_games=sub_w.rename(columns={"Team1":"WTeamID","Team2":"LTeamID"}).copy()
pred_games["WLoc"]="N"; pred_games["WScore"]=0; pred_games["LScore"]=0
pred_games["IsTourney"]=True
for col in [c for c in reg_df.columns if c not in pred_games.columns]:
    pred_games[col]=0

pred_matchup,_=build_matchup(pred_games,team_stats,recent,elo_dict,seed_feat)
X_pred=pred_matchup[feature_cols].fillna(0)
preds=model.predict_proba(X_pred)[:,1]

seed_map={}
for _,r in seeds_df[seeds_df["Season"]==2026].iterrows():
    seed_map[int(r.TeamID)]=parse_seed(r.Seed)
adj=preds.copy()
for i,(t1,t2) in enumerate(zip(pred_matchup["TeamA"].values,pred_matchup["TeamB"].values)):
    s1=seed_map.get(int(t1),8); s2=seed_map.get(int(t2),8)
    key=(min(s1,s2),max(s1,s2))
    if key in SEED_OVERRIDES:
        ov=SEED_OVERRIDES[key]
        if s1<s2: adj[i]=max(adj[i],ov)
        else:     adj[i]=min(adj[i],1-ov)

sub_w=sub_w.copy(); sub_w["Pred"]=adj.clip(0.025,0.975)
out=Path(".")/"submission_womens_2026_lgbm.csv"
sub_w[["ID","Pred"]].to_csv(out,index=False)

print("\n"+"="*60)
print("  FINAL RESULTS — WOMEN'S LightGBM")
print("="*60)
print(f"  Val  LogLoss  : {val_ll:.5f}")
print(f"  Test LogLoss  : {test_ll:.5f}")
print(f"  Test AUC      : {test_auc:.4f}")
print(f"  Test Brier    : {test_bri:.5f}")
print(f"  Test Accuracy : {test_acc:.4f}  ({n_correct}/{len(y_te)})")
print(f"  Runtime       : {mins()}")
print(f"  ✅ Saved → {out}")
print("="*60)
print(sub_w[["ID","Pred"]].head(10).to_string(index=False))

  MARCH MANIA 2026 — WOMEN'S  (89% EXACT REPLICA)

[1/6] Loading …  0.0m
   reg=81,708  tourney=961
[2/6] ELO …  0.0m
[3/6] Team features …  0.0m
[4/6] Matchups …  0.0m
   Rows=82,669  Features=142
   Train=77,158  Val=5,511  Test=67

[5/6] Training …  0.1m

   [Validation]    LogLoss : 0.31932
   [2025 Tourney]  LogLoss : 0.28562
   [2025 Tourney]  AUC     : 0.9649
   [2025 Tourney]  Accuracy: 0.8955  (60/67)

[6/6] Predicting 2026 …  0.9m
   Women's rows: 65,703

  FINAL RESULTS — WOMEN'S LightGBM
  Val  LogLoss  : 0.31932
  Test LogLoss  : 0.28562
  Test AUC      : 0.9649
  Test Brier    : 0.08614
  Test Accuracy : 0.8955  (60/67)
  Runtime       : 1.0m
  ✅ Saved → submission_womens_2026_lgbm.csv
            ID     Pred
2026_3101_3102 0.304888
2026_3101_3103 0.304888
2026_3101_3104 0.304888
2026_3101_3105 0.304888
2026_3101_3106 0.304888
2026_3101_3107 0.304888
2026_3101_3108 0.304888
2026_3101_3110 0.304888
2026_3101_3111 0.304888
2026_3101_3112 0.304888
